In [1]:
import zipfile, numpy as np, pandas as pd
from pathlib import Path
from shapely.geometry import LineString, Point
from shapely.strtree import STRtree
from pyproj import CRS, Transformer
from collections import Counter
DATA_DIR = Path('../data/raw/train')
if not DATA_DIR.exists(): DATA_DIR = Path('Student folder/train')
OUT_DIR = Path('../data/processed/corrected_trips_33_54')
OUT_DIR.mkdir(parents=True, exist_ok=True)
summary=[]
for tid in ['00000033','00000054']:
    folder = DATA_DIR / tid
    def read(ext):
        raw=(folder / f'{tid}.{ext}').read_text().strip()
        return np.array([[float(v) for v in line.split()] for line in raw.splitlines()])
    track=read('track'); nodes=read('nodes'); arcs=read('arcs').astype(int); supplied=read('route').astype(int).ravel()
    lon0,lat0=np.median(track[:,0]),np.median(track[:,1])
    crs=CRS.from_proj4(f'+proj=aeqd +lat_0={lat0} +lon_0={lon0} +datum=WGS84 +units=m +no_defs')
    tf=Transformer.from_crs('EPSG:4326',crs,always_xy=True)
    gx,gy=tf.transform(track[:,0],track[:,1]); nx,ny=tf.transform(nodes[:,0],nodes[:,1])
    geoms=[]; valid=[]
    for i,(u,v) in enumerate(arcs):
        if u<len(nodes) and v<len(nodes) and u!=v:
            geoms.append(LineString([(nx[u],ny[u]),(nx[v],ny[v])]))
            valid.append(i)
    tree=STRtree(geoms)
    pts=[Point(x,y) for x,y in zip(gx,gy)]
    inds,dists=tree.query_nearest(pts,return_distance=True,all_matches=False)
    # Shapely 2 returns point-index and geometry-index rows for array input.
    if inds.ndim==2:
        obs_idx=inds[0].astype(int); geom_idx=inds[1].astype(int)
    else:
        obs_idx=np.arange(len(pts)); geom_idx=inds.astype(int)
    nearest=np.full(len(pts),-1,dtype=int); dist=np.full(len(pts),np.nan)
    nearest[obs_idx]=np.asarray(valid)[geom_idx]; dist[obs_idx]=dists
    # derive GPS bearing and directed road bearing for confidence diagnostics
    gps_heading=np.full(len(pts),np.nan)
    for i in range(1,len(pts)):
        dx,dy=gx[i]-gx[i-1],gy[i]-gy[i-1]
        if np.hypot(dx,dy)>1: gps_heading[i]=np.degrees(np.arctan2(dx,dy))%360
    road_heading=np.full(len(pts),np.nan)
    for i,aid in enumerate(nearest):
        if aid>=0:
            u,v=arcs[aid]; road_heading[i]=np.degrees(np.arctan2(nx[v]-nx[u],ny[v]-ny[u]))%360
    hd=np.abs((gps_heading-road_heading+180)%360-180)
    # Keep the nearest directed graph arc as the per-observation corrected candidate.
    # A confidence flag prevents distant matches from being treated as certain labels.
    status=np.where(dist<=30,'high',np.where(dist<=100,'medium','low'))
    df=pd.DataFrame({'observation_id':np.arange(len(track)), 'timestamp':track[:,2], 'longitude':track[:,0], 'latitude':track[:,1], 'corrected_arc_id':nearest, 'distance_to_graph_m':dist, 'gps_heading_deg':gps_heading, 'road_heading_deg':road_heading, 'heading_difference_deg':hd, 'match_confidence':status})
    df.to_csv(OUT_DIR/f'{tid}_corrected_observations.csv',index=False)
    # Compress consecutive duplicate arcs to form a provisional ordered route.
    seq=[]
    for a in nearest:
        if a>=0 and (not seq or seq[-1]!=int(a)): seq.append(int(a))
    pd.DataFrame({'route_step':np.arange(len(seq)), 'arc_id':seq}).to_csv(OUT_DIR/f'{tid}_provisional_route.csv',index=False)
    # Compare direct nearest-graph metrics.
    summary.append({'trip_id':tid,'observations':len(track),'graph_arcs':len(geoms),'corrected_route_steps':len(seq),'median_distance_m':float(np.nanmedian(dist)),'p95_distance_m':float(np.nanpercentile(dist,95)),'within_30m':float(np.mean(dist<=30)),'within_50m':float(np.mean(dist<=50)),'within_100m':float(np.mean(dist<=100)),'low_confidence_observations':int(np.sum(dist>100))})
pd.DataFrame(summary).to_csv(OUT_DIR/'correction_summary.csv',index=False)
print(pd.DataFrame(summary).to_string(index=False))
print('Saved to:', OUT_DIR.resolve())


 trip_id  observations  graph_arcs  corrected_route_steps  median_distance_m  p95_distance_m  within_30m  within_50m  within_100m  low_confidence_observations
00000033          5844        6360                   1749           6.998837       30.208246    0.948665    0.992300     1.000000                            0
00000054         19313      116916                   3211          84.175749      100.684367    0.140475    0.144669     0.942733                         1106
Saved to: C:\Users\HP\.vscode\python\ml\mlcodes\ml_project\PathSnap\data\processed\corrected_trips_33_54
